# Introducción a MLOps y Herramientas Cloud

_De Jupyter en tu laptop a un modelo corriendo en la nube_

**Módulo 4 — MLOps y Cloud** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

## 1. ¿Qué es MLOps?

**MLOps** es la disciplina que aplica las prácticas de DevOps (automatización, CI/CD, observabilidad, infraestructura como código) al ciclo de vida de los modelos de Machine Learning.

El objetivo es responder, de forma reproducible y automatizada, a cuatro preguntas:

1. **¿Cómo entreno** este modelo otra vez, con datos nuevos, sin tocar nada a mano?
2. **¿Cómo lo despliego** a producción para que reciba tráfico real?
3. **¿Cómo sé** que está funcionando bien una vez desplegado?
4. **¿Cómo lo rollbackeo** si algo sale mal?

Un notebook de Jupyter responde a la pregunta 1 una sola vez, en la laptop de una persona. MLOps responde a las cuatro, en cualquier momento, sin intervención humana.

## 2. El ciclo de vida del modelo

```
  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
  │  Datos   │───▶│  Entreno │───▶│ Registro │───▶│ Despliego│───▶│ Monitoreo│
  └──────────┘    └──────────┘    └──────────┘    └──────────┘    └────┬─────┘
       ▲                                                                │
       └────────────────────── retrain trigger ◀───────────────────────┘
```

Cada flecha es una pieza de software que en MLOps se automatiza:

- **Datos → Entreno**: pipelines de datos (Airflow, Prefect), object storage (Blob / S3 / GCS).
- **Entreno → Registro**: experiment tracking (MLflow, W&B), model registry, versionado de artefactos.
- **Registro → Despliego**: containerización (Docker), orquestación (Kubernetes, Azure Container Apps, AWS ECS), serving (FastAPI, BentoML, KServe).
- **Despliego → Monitoreo**: métricas (Prometheus / Azure Monitor), drift detection (Evidently), logging estructurado.
- **Monitoreo → Datos**: alertas que disparan un retrain automático.

## 3. Mapa de herramientas cloud que veremos en este módulo

| Capa | Concepto | Herramienta cubierta |
|---|---|---|
| Almacenamiento de datos / modelos | Object Storage | **Azure Blob Storage** (notebook 02) |
| Empaquetado del entrenamiento | Container | **Docker** (`docker-training/`) |
| Distribución de la imagen | Container Registry | **GitHub Container Registry** (`ghcr.io`) vía GitHub Actions |
| Automatización del build | CI/CD | **GitHub Actions** (`.github/workflows/`) |
| Infraestructura | IaC + cómputo | **Terraform** + **Azure VM** (`terraform/`) |
| Bootstrap de proyectos | Generador de plantillas | **Skill de Claude Code** (`skills/ds-ml-repo-init/`) |

## 4. ¿Por qué Azure?

No hay diferencia conceptual entre los tres grandes (AWS / Azure / GCP): los tres ofrecen blobs, VMs, contenedores y registries. En este módulo usamos **Azure** porque:

- La capa gratuita incluye 12 meses con `B1s` VMs y 5 GB de Blob Storage.
- El SDK de Python (`azure-storage-blob`) es directo y bien documentado.
- El portal es relativamente simple para principiantes.

Los conceptos (object storage, IaC, contenedores) se traducen 1:1 a AWS (S3 + EC2 + ECR) o GCP (GCS + Compute Engine + Artifact Registry). El skill `ds-ml-repo-init` incluye snippets para los tres.

## 5. Lo que NO es MLOps

Tres confusiones comunes:

- **MLOps ≠ tener un Jupyter Notebook en la nube.** Subir un `.ipynb` a Azure ML Studio no es MLOps. Si una persona tiene que hacer click en "Run All" para reentrenar, no es MLOps.
- **MLOps ≠ Kubernetes obligatorio.** Muchos equipos exitosos corren sus modelos en una VM con `docker run` o en serverless. La complejidad debe ser proporcional al problema.
- **MLOps ≠ Data Engineering.** Los pipelines de datos son un pre-requisito, pero MLOps cubre específicamente el ciclo del **modelo** (entreno, versionado, despliegue, monitoreo). No el ETL.

## 6. El "buen ciudadano" del repo ML

Antes de hablar de Docker o Terraform, hay un paso anterior: **tener un repositorio bien estructurado**. La mayoría de los problemas de MLOps son consecuencia de un repo desordenado.

Las reglas básicas son:

1. **Datos fuera de git.** Los CSV nunca se commitean.
2. **Modelos fuera de git.** Los `.pkl` se guardan en Blob Storage / S3 / GCS, no en `git lfs` y mucho menos en `main`.
3. **Código en `src/` empaquetado.** Imports relativos sobreviven a renombres; scripts sueltos no.
4. **Notebooks solo para exploración.** Lo que sirve se migra a `src/`.
5. **Secretos en variables de entorno.** Nunca hardcodeados, nunca commiteados.

El skill `skills/ds-ml-repo-init/` automatiza esto: con un solo prompt a Claude Code, genera un repositorio completo con esta estructura.

👉 Lee `skills/ds-ml-repo-init/SKILL.md` para entender cómo se invoca.

## 7. Lo que vamos a construir en este módulo

Al terminar el módulo vas a tener este pipeline funcionando **de verdad**, no en un slide:

```
  [ tú haces git push ]
          │
          ▼
  ┌───────────────────┐
  │  GitHub Actions   │  build-modulo4-image.yml
  │  build + push     │  → ghcr.io/<user>/dsrp-modulo4-trainer:latest
  └────────┬──────────┘
           │
           ▼
  ┌───────────────────┐    docker pull     ┌─────────────────────┐
  │  Azure VM B1s     │ ◀───────────────── │  ghcr.io (registry) │
  │  (Terraform IaC)  │                    └─────────────────────┘
  │  docker run ...   │
  └────────┬──────────┘
           │                                ┌─────────────────────┐
           ├── pull Telco.csv ────────────▶ │  Azure Blob Storage │
           │                                │  container=models   │
           └── push modelo_telco.pkl ─────▶ └─────────────────────┘
```

**Próximo paso**: abre `02_azure_blob_storage_sdk.ipynb` para empezar a hablar con Azure Blob Storage desde Python.